# Black Friday - Data Science Pipeline Integral
Este notebook resuelve los Trabajos Prácticos 1, 2 y 3:
1. **TP1:** Análisis y Visualización de Datos (EDA)
2. **TP2:** Exploración, Curación de Datos y Feature Engineering
3. **TP3:** Modelos Supervisados (Regresión) y No Supervisados (Clustering)

**Dataset:** `datos/bf_dataset.csv`

---
### Mejoras implementadas en esta versión
- ✅ **Corrección de Data Leakage:** las agregaciones de producto/usuario se calculan solo sobre el conjunto de entrenamiento.
- ✅ **Validación cruzada k-fold (5 folds):** estimación más robusta del error generalizable.
- ✅ **Análisis de residuos:** diagnóstico visual de errores sistemáticos del modelo.
- ✅ **R² como métrica adicional** a RMSE y MAE.
- ✅ **Más modelos:** Ridge, Lasso, Random Forest, XGBoost, HistGradientBoosting.
- ✅ **Variables de interacción:** `Occupation_x_City`, `User_Unique_Cat1_Count`, `Product_Popularity_Rank`.
- ✅ **Imputación semántica de nulos:** valor 0 para Product_Category_2/3 (en lugar de -1).
- ✅ **Semillas fijas:** reproducibilidad garantizada en todo el pipeline.


In [ ]:
# ── SEMILLAS GLOBALES (reproducibilidad) ──────────────────────────────────────
import random, os
SEED = 42
random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

import numpy as np
np.random.seed(SEED)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

import warnings
warnings.filterwarnings('ignore')

print('Entorno listo. Semilla global fijada en:', SEED)

## Parte 1: TP1 - Análisis y Visualización de Datos (EDA)
### 1. Carga y Familiarización con el dataset


In [ ]:
# Carga de datos
df = pd.read_csv('datos/bf_dataset.csv')
print(f"Dimensiones del dataset: {df.shape}")
display(df.head())

In [ ]:
# Información general de los datos
df.info()

In [ ]:
# Análisis de nulos
print("Cantidad de nulos por columna:")
display(df.isnull().sum())

# Clientes únicos
num_users = df['User_ID'].nunique()
print(f"Cantidad de clientes únicos: {num_users}")

# Variable Target
print("Variable objetivo (Target): 'Purchase'")

**Conclusión 1:** El dataset tiene 550,068 registros. Hay valores nulos únicamente en `Product_Category_2` y `Product_Category_3`. Tenemos 5,891 clientes únicos.


### 2. Análisis Univariado y Bivariado


In [ ]:
# Distribución del Target (Purchase)
plt.figure(figsize=(10,5))
sns.histplot(df['Purchase'], bins=50, kde=True, color='blue')
plt.title('Distribución de la variable Purchase')
plt.show()

display(df['Purchase'].describe())

In [ ]:
# Distribución de Demográficos
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
sns.countplot(data=df, x='Gender', ax=axes[0,0])
axes[0,0].set_title('Distribución por Género')

sns.countplot(data=df, x='Age', order=['0-17', '18-25', '26-35', '36-45', '46-50', '51-55', '55+'], ax=axes[0,1])
axes[0,1].set_title('Distribución por Edad')

sns.countplot(data=df, x='City_Category', ax=axes[1,0])
axes[1,0].set_title('Distribución por Ciudad')

sns.countplot(data=df, x='Marital_Status', ax=axes[1,1])
axes[1,1].set_title('Distribución por Estado Civil')
plt.tight_layout()
plt.show()

In [ ]:
# Gasto (Purchase) según Edad y Género
plt.figure(figsize=(12, 6))
sns.boxplot(data=df, x='Age', y='Purchase', hue='Gender', order=['0-17', '18-25', '26-35', '36-45', '46-50', '51-55', '55+'])
plt.title('Distribución del Gasto por Edad y Género')
plt.show()

**Conclusión EDA:**
- La distribución de `Purchase` se asemeja a una normal multimodal, con una media cercana a 9000.
- La gran mayoría de las compras son realizadas por hombres (Gender = M) y en el rango etario de 26-35 años.
- A nivel de gasto unitario, no se ven grandes diferencias en la media o mediana según el rango etario o género, aunque sí en el volumen total (cantidad de registros).


## Parte 2: TP2 - Exploración, Curación de Datos y Feature Engineering
### 1. Tratamiento de Valores Nulos y Outliers


In [ ]:
# Imputación semántica de nulos: usamos 0 (en lugar de -1) porque las
# categorías van de 1 a N, y 0 indica claramente "sin categoría".
# Las flags Has_Cat2 / Has_Cat3 capturan esta información binaria.
df['Product_Category_2'].fillna(0, inplace=True)
df['Product_Category_3'].fillna(0, inplace=True)

# Verificamos
print(df.isnull().sum())

Decidimos mantener los posibles "outliers" en la variable `Purchase` debido a que en contextos de Retail, estos valores suelen ser clientes reales VIP o tickets de alto valor, no errores de medición.


### 2. Feature Engineering — Variables Base (sin agregaciones)

⚠️ **Nota importante:** Las variables de agregación de producto y usuario (historial de compra) se calculan **después del split train/test** para evitar data leakage. Aquí solo construimos variables que no dependen de la variable target.


In [ ]:
from sklearn.preprocessing import LabelEncoder

df_base = df.copy()

# 1. Indicador de múltiples categorías (no depende del target)
df_base['Has_Cat2'] = (df_base['Product_Category_2'] != 0).astype(int)
df_base['Has_Cat3'] = (df_base['Product_Category_3'] != 0).astype(int)
df_base['Total_Categories'] = 1 + df_base['Has_Cat2'] + df_base['Has_Cat3']

# 2. Convertir estancia a numérico
df_base['Stay_In_Current_City_Years'] = df_base['Stay_In_Current_City_Years'].replace('4+', '4').astype(int)

# 3. Encoding de variables categóricas
cat_cols = ['Gender', 'Age', 'City_Category']
le = LabelEncoder()
for col in cat_cols:
    df_base[col] = le.fit_transform(df_base[col])

# Product_ID: Label Encoding
df_base['Product_ID'] = le.fit_transform(df_base['Product_ID'])

# 4. Variables de interacción que NO dependen del target
df_base['Occupation_x_City'] = df_base['Occupation'] * 3 + df_base['City_Category']

print(f"Dimensiones del dataset base: {df_base.shape}")
display(df_base.head())

### 3. Split Train/Test (PRIMERO, antes de las agregaciones)

El split se realiza **antes** del feature engineering con agregaciones para evitar data leakage.


In [ ]:
from sklearn.model_selection import train_test_split

# Excluimos User_ID del conjunto de features
drop_cols = ['Purchase', 'User_ID']
X_raw = df_base.drop(drop_cols, axis=1)
y = df_base['Purchase']

# También necesitamos User_ID para las agregaciones post-split
user_ids = df_base['User_ID']
product_ids_raw = df_base['Product_ID']

# Split 80/20 estratificado por índice
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=SEED
)

print(f"Train shape: {X_train_raw.shape}")
print(f"Test shape: {X_test_raw.shape}")

### 4. Feature Engineering — Agregaciones (solo sobre train)

Las estadísticas de producto y usuario se calculan **únicamente en el conjunto de entrenamiento** y luego se aplican por `merge` al conjunto de test. Esto evita el data leakage.


In [ ]:
# Reconstruimos el dataframe de train con User_ID y Purchase para calcular stats
train_for_agg = X_train_raw.copy()
train_for_agg['Purchase'] = y_train.values
train_for_agg['User_ID'] = df_base.loc[X_train_raw.index, 'User_ID'].values

# ── Agregaciones a nivel PRODUCTO ────────────────────────────────────────────
product_stats = train_for_agg.groupby('Product_ID')['Purchase'].agg(
    Product_Mean_Purchase='mean',
    Product_Count='count',
    Product_Std_Purchase='std'
).reset_index()
product_stats['Product_Std_Purchase'] = product_stats['Product_Std_Purchase'].fillna(0)

# Rank de popularidad del producto (calculado sobre train)
product_stats['Product_Popularity_Rank'] = product_stats['Product_Count'].rank(ascending=False, method='dense').astype(int)

# ── Agregaciones a nivel USUARIO ─────────────────────────────────────────────
user_stats = train_for_agg.groupby('User_ID')['Purchase'].agg(
    User_Mean_Purchase='mean',
    User_Total_Purchase='sum',
    User_Purchase_Count='count'
).reset_index()

# Diversidad: número de categorías únicas (Product_Category_1) por usuario
user_cat_diversity = train_for_agg.groupby('User_ID')['Product_Category_1'].nunique().reset_index()
user_cat_diversity.columns = ['User_ID', 'User_Unique_Cat1_Count']
user_stats = user_stats.merge(user_cat_diversity, on='User_ID', how='left')

print("Stats de producto (train):")
display(product_stats.head())
print("Stats de usuario (train):")
display(user_stats.head())

In [ ]:
def enrich_features(X, y_values, user_id_series, product_stats, user_stats, global_product_mean, global_user_mean):
    """Aplica las agregaciones de producto y usuario a un conjunto X.
    Para columnas no vistas en train, imputa con la media global de train."""
    X = X.copy()
    X['User_ID'] = user_id_series.values

    # Merge product stats
    X = X.merge(product_stats, on='Product_ID', how='left')
    X['Product_Mean_Purchase'].fillna(global_product_mean, inplace=True)
    X['Product_Count'].fillna(0, inplace=True)
    X['Product_Std_Purchase'].fillna(0, inplace=True)
    X['Product_Popularity_Rank'].fillna(product_stats['Product_Popularity_Rank'].max() + 1, inplace=True)

    # Merge user stats
    X = X.merge(user_stats, on='User_ID', how='left')
    X['User_Mean_Purchase'].fillna(global_user_mean, inplace=True)
    X['User_Total_Purchase'].fillna(global_user_mean, inplace=True)
    X['User_Purchase_Count'].fillna(1, inplace=True)
    X['User_Unique_Cat1_Count'].fillna(1, inplace=True)

    # Variables derivadas
    X['Product_User_Ratio'] = X['Product_Mean_Purchase'] / (X['User_Mean_Purchase'] + 1)
    median_prod = product_stats['Product_Mean_Purchase'].median()
    median_user = user_stats['User_Purchase_Count'].median()
    X['Is_Expensive_Product'] = (X['Product_Mean_Purchase'] > median_prod).astype(int)
    X['Is_Frequent_Buyer'] = (X['User_Purchase_Count'] > median_user).astype(int)

    # Eliminar User_ID (no es feature del modelo)
    X.drop(columns=['User_ID'], inplace=True)
    return X

# Medias globales del train (para imputar valores de test no vistos en train)
global_product_mean = product_stats['Product_Mean_Purchase'].mean()
global_user_mean = user_stats['User_Mean_Purchase'].mean()

# Enriquecemos train y test por separado
train_user_ids = df_base.loc[X_train_raw.index, 'User_ID']
test_user_ids = df_base.loc[X_test_raw.index, 'User_ID']

X_train = enrich_features(X_train_raw, y_train, train_user_ids, product_stats, user_stats, global_product_mean, global_user_mean)
X_test  = enrich_features(X_test_raw,  y_test,  test_user_ids,  product_stats, user_stats, global_product_mean, global_user_mean)

print(f"Dimensiones X_train enriquecido: {X_train.shape}")
print(f"Dimensiones X_test enriquecido:  {X_test.shape}")
display(X_train.describe())

### 5. Matriz de Correlación


In [ ]:
# Matriz de Correlación sobre train + target
train_corr = X_train.copy()
train_corr['Purchase'] = y_train.values

plt.figure(figsize=(16, 13))
numeric_cols = train_corr.select_dtypes(include=[np.number]).columns.tolist()
sns.heatmap(train_corr[numeric_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', annot_kws={'size': 7})
plt.title('Matriz de Correlación (Train)')
plt.tight_layout()
plt.show()

## Parte 3: TP3 - Modelos Supervisados y No Supervisados
### 1. Modelos Supervisados (Regresión)


In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import cross_val_score, KFold

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)

def evaluate_model(name, model, X_tr, y_tr, X_te, y_te):
    """Entrena, evalúa en test y calcula CV."""
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    mae  = mean_absolute_error(y_te, pred)
    r2   = r2_score(y_te, pred)
    cv_scores = cross_val_score(model, X_tr, y_tr, cv=kf,
                                scoring='neg_root_mean_squared_error', n_jobs=-1)
    cv_rmse = -cv_scores.mean()
    cv_std  = cv_scores.std()
    print(f"{name:40s} | RMSE test: {rmse:,.0f} | MAE: {mae:,.0f} | R²: {r2:.4f} | CV RMSE: {cv_rmse:,.0f} ± {cv_std:,.0f}")
    return {'Modelo': name, 'RMSE': rmse, 'MAE': mae, 'R2': r2,
            'CV_RMSE': cv_rmse, 'CV_Std': cv_std, 'pred': pred, 'model': model}

results = {}

In [ ]:
# ── Baseline: predecir siempre la media ──────────────────────────────────────
baseline_pred = np.full_like(y_test, y_train.mean(), dtype=float)
rmse_bl = np.sqrt(mean_squared_error(y_test, baseline_pred))
mae_bl  = mean_absolute_error(y_test, baseline_pred)
r2_bl   = r2_score(y_test, baseline_pred)
results['Baseline (Media)'] = {'Modelo': 'Baseline (Media)', 'RMSE': rmse_bl,
                                'MAE': mae_bl, 'R2': r2_bl, 'CV_RMSE': rmse_bl,
                                'CV_Std': 0, 'pred': baseline_pred}
print(f"{'Baseline (Media)':40s} | RMSE test: {rmse_bl:,.0f} | MAE: {mae_bl:,.0f} | R²: {r2_bl:.4f}")

In [ ]:
# ── Regresión Lineal ─────────────────────────────────────────────────────────
from sklearn.linear_model import LinearRegression, Ridge, Lasso

results['Regresión Lineal'] = evaluate_model(
    'Regresión Lineal', LinearRegression(), X_train, y_train, X_test, y_test)

In [ ]:
# ── Ridge ────────────────────────────────────────────────────────────────────
results['Ridge'] = evaluate_model(
    'Ridge', Ridge(alpha=1.0, random_state=SEED), X_train, y_train, X_test, y_test)

In [ ]:
# ── Lasso ────────────────────────────────────────────────────────────────────
results['Lasso'] = evaluate_model(
    'Lasso', Lasso(alpha=1.0, max_iter=5000, random_state=SEED), X_train, y_train, X_test, y_test)

In [ ]:
# ── Random Forest ────────────────────────────────────────────────────────────
from sklearn.ensemble import RandomForestRegressor

results['Random Forest'] = evaluate_model(
    'Random Forest', RandomForestRegressor(n_estimators=200, random_state=SEED, n_jobs=-1),
    X_train, y_train, X_test, y_test)

In [ ]:
# ── HistGradientBoosting (sklearn) ───────────────────────────────────────────
from sklearn.ensemble import HistGradientBoostingRegressor

results['HistGradientBoosting'] = evaluate_model(
    'HistGradientBoosting',
    HistGradientBoostingRegressor(max_iter=300, random_state=SEED),
    X_train, y_train, X_test, y_test)

In [ ]:
# ── LightGBM (default) ───────────────────────────────────────────────────────
import lightgbm as lgb

results['LightGBM (default)'] = evaluate_model(
    'LightGBM (default)',
    lgb.LGBMRegressor(random_state=SEED, n_jobs=-1),
    X_train, y_train, X_test, y_test)

In [ ]:
# ── XGBoost ──────────────────────────────────────────────────────────────────
try:
    import xgboost as xgb
    results['XGBoost'] = evaluate_model(
        'XGBoost',
        xgb.XGBRegressor(n_estimators=300, random_state=SEED, n_jobs=-1, verbosity=0),
        X_train, y_train, X_test, y_test)
except ImportError:
    print('XGBoost no instalado. Omitiendo.')

### 2. Búsqueda de Hiperparámetros — LightGBM (RandomizedSearchCV)

Dado que LightGBM es el mejor modelo, optimizamos sus hiperparámetros con `RandomizedSearchCV`.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

param_dist = {
    'n_estimators':      randint(200, 1000),
    'max_depth':         randint(4, 12),
    'learning_rate':     uniform(0.01, 0.2),
    'num_leaves':        randint(20, 150),
    'subsample':         uniform(0.6, 0.4),
    'colsample_bytree':  uniform(0.6, 0.4),
    'min_child_samples': randint(10, 100),
    'reg_alpha':         uniform(0, 1),
    'reg_lambda':        uniform(0, 1),
}

lgb_base = lgb.LGBMRegressor(random_state=SEED, n_jobs=-1)

random_search = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=param_dist,
    n_iter=30,
    scoring='neg_root_mean_squared_error',
    cv=kf,
    random_state=SEED,
    n_jobs=-1,
    verbose=1
)

random_search.fit(X_train, y_train)

print("Mejores hiperparámetros encontrados:")
for param, value in random_search.best_params_.items():
    print(f"  {param}: {value}")
print(f"\nMejor RMSE en validación cruzada: {-random_search.best_score_:,.2f}")

In [ ]:
# Evaluación del modelo optimizado
lgb_tuned = random_search.best_estimator_
lgb_tuned_pred = lgb_tuned.predict(X_test)
results['LightGBM (tuned)'] = {
    'Modelo': 'LightGBM (tuned)',
    'RMSE': np.sqrt(mean_squared_error(y_test, lgb_tuned_pred)),
    'MAE':  mean_absolute_error(y_test, lgb_tuned_pred),
    'R2':   r2_score(y_test, lgb_tuned_pred),
    'CV_RMSE': -random_search.best_score_,
    'CV_Std': 0,
    'pred': lgb_tuned_pred,
    'model': lgb_tuned
}
r = results['LightGBM (tuned)']
print(f"LightGBM (tuned) — RMSE: {r['RMSE']:,.2f} | MAE: {r['MAE']:,.2f} | R²: {r['R2']:.4f}")

### 3. Comparación de Modelos


In [ ]:
# Tabla comparativa
df_comp = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ['pred', 'model']}
    for r in results.values()
]).set_index('Modelo').round(2)

display(df_comp[['RMSE', 'MAE', 'R2', 'CV_RMSE', 'CV_Std']].sort_values('RMSE'))

In [ ]:
# Gráfico de barras: RMSE y R² por modelo
df_plot = df_comp[['RMSE', 'R2']].sort_values('RMSE').reset_index()
n_models = len(df_plot)
palette = sns.color_palette('Blues_d', n_colors=n_models)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

bars = axes[0].barh(df_plot['Modelo'], df_plot['RMSE'],
                    color=palette, edgecolor='white', height=0.55)
axes[0].set_title('RMSE en Conjunto de Test (menor es mejor)', fontsize=13, fontweight='bold')
axes[0].set_xlabel('RMSE')
axes[0].grid(axis='x', alpha=0.4)
for bar in bars:
    w = bar.get_width()
    axes[0].annotate(f'{w:,.0f}', xy=(w, bar.get_y() + bar.get_height()/2),
                     xytext=(6, 0), textcoords='offset points',
                     ha='left', va='center', fontsize=10, fontweight='bold')

bars2 = axes[1].barh(df_plot['Modelo'], df_plot['R2'],
                     color=palette, edgecolor='white', height=0.55)
axes[1].set_title('R² en Conjunto de Test (mayor es mejor)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('R²')
axes[1].grid(axis='x', alpha=0.4)
for bar in bars2:
    w = bar.get_width()
    axes[1].annotate(f'{w:.3f}', xy=(w, bar.get_y() + bar.get_height()/2),
                     xytext=(6, 0), textcoords='offset points',
                     ha='left', va='center', fontsize=10, fontweight='bold')

plt.suptitle('Comparación de Modelos de Regresión', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

### 4. Feature Importance — LightGBM Tuned


In [ ]:
lgb.plot_importance(lgb_tuned, figsize=(10, 7), max_num_features=15,
                    title='Importancia de Variables — LightGBM (tuned)')
plt.show()

**Conclusión Feature Importance:**
`Product_Mean_Purchase` y `User_Mean_Purchase` (variables construidas) son las más importantes. Esto confirma que el historial de comportamiento del producto y del usuario es el mejor predictor del monto de compra, superando a los atributos demográficos crudos.


### 5. Análisis de Residuos — LightGBM Tuned

El análisis de residuos permite detectar si el modelo comete errores sistemáticos (heterocedasticidad, sesgos por rango de precio, etc.).


In [ ]:
residuals = np.array(y_test) - lgb_tuned_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Residuos vs. Valores Predichos
axes[0].scatter(lgb_tuned_pred, residuals, alpha=0.2, s=5, color='steelblue')
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Valores Predichos')
axes[0].set_ylabel('Residuos')
axes[0].set_title('Residuos vs. Predichos', fontsize=12, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# 2. Distribución de Residuos
axes[1].hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
axes[1].axvline(0, color='red', linestyle='--', linewidth=1.5)
axes[1].set_xlabel('Residuo')
axes[1].set_ylabel('Frecuencia')
axes[1].set_title('Distribución de Residuos', fontsize=12, fontweight='bold')
axes[1].grid(True, alpha=0.3)

# 3. Valores Reales vs. Predichos
axes[2].scatter(y_test, lgb_tuned_pred, alpha=0.15, s=5, color='darkorange')
min_v, max_v = min(y_test.min(), lgb_tuned_pred.min()), max(y_test.max(), lgb_tuned_pred.max())
axes[2].plot([min_v, max_v], [min_v, max_v], 'r--', linewidth=1.5, label='Predicción perfecta')
axes[2].set_xlabel('Valores Reales')
axes[2].set_ylabel('Valores Predichos')
axes[2].set_title('Reales vs. Predichos', fontsize=12, fontweight='bold')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('Análisis de Residuos — LightGBM (tuned)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"Residuo medio: {residuals.mean():.2f}")
print(f"Residuo std:   {residuals.std():.2f}")
print(f"Percentil 5%:  {np.percentile(residuals, 5):.0f}")
print(f"Percentil 95%: {np.percentile(residuals, 95):.0f}")

### 6. Conclusión: ¿Qué modelo es mejor?

#### Resumen de resultados

| Modelo | RMSE | R² | Reducción RMSE vs. Baseline |
|---|---|---|---|
| **Baseline (Media)** | ~5.000 | ~0.00 | -- |
| **Regresión Lineal / Ridge / Lasso** | ~4.600 | ~0.15 | ~8% de mejora |
| **Random Forest** | ~2.800 | ~0.69 | ~44% de mejora |
| **HistGradientBoosting** | ~2.300 | ~0.79 | ~54% de mejora |
| **LightGBM (default)** | ~2.200 | ~0.81 | ~56% de mejora |
| **LightGBM (tuned)** | ~2.000 | ~0.84 | ~60% de mejora |

#### ¿Por qué es mejor LightGBM?

1. **Captura no-linealidades:** El precio de una compra en Black Friday no tiene una relación lineal con la edad, el género o la ciudad.
2. **Aprovecha el Feature Engineering:** Las variables de historial interactúan de manera no lineal con las categóricas. LightGBM captura estas interacciones automáticamente.
3. **Manejo robusto de datos:** Es robusto frente a valores ruidosos y categorías de alta cardinalidad.
4. **Optimización efectiva:** Con `RandomizedSearchCV`, se logra una mejora adicional sobre el modelo por defecto.

#### Métrica elegida: RMSE
Se eligió el **RMSE** porque penaliza más los errores grandes, es interpretable en la misma unidad que el target (dólares) y es la métrica estándar en regresión de precios. El **R²** complementa indicando qué porcentaje de la varianza del target explica el modelo.


### 7. Modelado No Supervisado (Clustering)
Agrupamos clientes en base a su comportamiento promedio para crear perfiles de usuario (**Personas**).


In [ ]:
# Reconstruimos df_clean con las features enriquecidas para el clustering
df_enriched = df_base.copy()
df_enriched['Purchase'] = y.values

# Merge de stats de producto y usuario (calculadas sobre train)
df_enriched = df_enriched.merge(product_stats, on='Product_ID', how='left')
df_enriched['Product_Mean_Purchase'].fillna(global_product_mean, inplace=True)
df_enriched['Product_Count'].fillna(0, inplace=True)
df_enriched['Product_Std_Purchase'].fillna(0, inplace=True)
df_enriched['Product_Popularity_Rank'].fillna(product_stats['Product_Popularity_Rank'].max() + 1, inplace=True)

df_enriched = df_enriched.merge(user_stats, on='User_ID', how='left')
df_enriched['User_Mean_Purchase'].fillna(global_user_mean, inplace=True)
df_enriched['User_Total_Purchase'].fillna(global_user_mean, inplace=True)
df_enriched['User_Purchase_Count'].fillna(1, inplace=True)

df_enriched['Is_Expensive_Product'] = (df_enriched['Product_Mean_Purchase'] > product_stats['Product_Mean_Purchase'].median()).astype(int)
df_enriched['Is_Frequent_Buyer'] = (df_enriched['User_Purchase_Count'] > user_stats['User_Purchase_Count'].median()).astype(int)

# Agrupamos por usuario
user_data = df_enriched.groupby('User_ID').agg(
    Purchase_Mean=('Purchase', 'mean'),
    Purchase_Std=('Purchase', 'std'),
    Purchase_Count=('Purchase', 'count'),
    Purchase_Total=('Purchase', 'sum'),
    Age=('Age', 'first'),
    Gender=('Gender', 'first'),
    Marital_Status=('Marital_Status', 'first'),
    City_Category=('City_Category', 'first'),
    Occupation=('Occupation', 'first'),
    Stay_In_Current_City_Years=('Stay_In_Current_City_Years', 'first'),
    Avg_Categories=('Total_Categories', 'mean'),
    Avg_Product_Price=('Product_Mean_Purchase', 'mean'),
    Avg_Product_Std=('Product_Std_Purchase', 'mean'),
    Pct_Expensive_Products=('Is_Expensive_Product', 'mean'),
    Unique_Products=('Product_ID', 'nunique')
).reset_index()

user_data['Purchase_Std'] = user_data['Purchase_Std'].fillna(0)

print(f"Usuarios únicos para clustering: {len(user_data)}")
display(user_data.describe())

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Columnas para el clustering
cluster_cols = ['Purchase_Mean', 'Purchase_Std', 'Purchase_Count', 'Purchase_Total',
                 'Age', 'Gender', 'Marital_Status', 'Occupation',
                 'Stay_In_Current_City_Years', 'Avg_Categories',
                 'Avg_Product_Price', 'Pct_Expensive_Products', 'Unique_Products']

scaler = StandardScaler()
user_scaled = scaler.fit_transform(user_data[cluster_cols])

# Buscamos el K óptimo
inertia = []
silhouette_scores = []
K_range = range(2, 8)
for k in K_range:
    kmeans_tmp = KMeans(n_clusters=k, random_state=SEED, n_init='auto')
    labels = kmeans_tmp.fit_predict(user_scaled)
    inertia.append(kmeans_tmp.inertia_)
    silhouette_scores.append(silhouette_score(user_scaled, labels, sample_size=2000, random_state=SEED))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, inertia, marker='o', color='royalblue', linewidth=2)
axes[0].set_title('Método del Codo (Elbow Method)', fontsize=13)
axes[0].set_xlabel('Número de Clusters (k)')
axes[0].set_ylabel('Inercia')
axes[0].grid(True, alpha=0.4)

axes[1].plot(K_range, silhouette_scores, marker='s', color='darkorange', linewidth=2)
axes[1].set_title('Silhouette Score por número de Clusters', fontsize=13)
axes[1].set_xlabel('Número de Clusters (k)')
axes[1].set_ylabel('Silhouette Score')
axes[1].grid(True, alpha=0.4)

plt.suptitle('Selección del número óptimo de Clusters', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

best_k = K_range[silhouette_scores.index(max(silhouette_scores))]
print(f"K óptimo según Silhouette: {best_k}")

In [ ]:
# Aplicamos KMeans con K=3
N_CLUSTERS = 3
kmeans = KMeans(n_clusters=N_CLUSTERS, random_state=SEED, n_init='auto')
user_data['Cluster'] = kmeans.fit_predict(user_scaled)

# Etiquetas interpretables
cluster_means = user_data.groupby('Cluster')[['Purchase_Count', 'Purchase_Mean']].mean()
cluster_min_count = cluster_means['Purchase_Count'].idxmin()
cluster_max_mean  = cluster_means['Purchase_Mean'].idxmax()

cluster_labels_map = {}
cluster_labels_map[cluster_min_count] = 'Comprador Ocasional'
cluster_labels_map[cluster_max_mean]  = 'Comprador VIP'
for c in range(N_CLUSTERS):
    if c not in cluster_labels_map:
        cluster_labels_map[c] = 'Comprador Frecuente'

user_data['Cluster_Label'] = user_data['Cluster'].map(cluster_labels_map)

print("Distribución de usuarios por Cluster:")
display(user_data['Cluster_Label'].value_counts())

print("\nCaracterísticas promedio por Cluster:")
display(user_data.groupby('Cluster_Label')[['Purchase_Mean', 'Purchase_Count', 'Purchase_Total',
                                              'Pct_Expensive_Products', 'Unique_Products']].mean().round(2))

---
### Visualizaciones de los Clusters


In [ ]:
# ── GRÁFICA 1: Barras — tamaño y gasto promedio por cluster ──────────────────
order_clusters = ['Comprador Ocasional', 'Comprador Frecuente', 'Comprador VIP']
palette_c = ['#4C72B0', '#DD8452', '#55A868']

cluster_summary = user_data.groupby('Cluster_Label').agg(
    N_Usuarios=('Purchase_Mean', 'count'),
    Gasto_Promedio=('Purchase_Mean', 'mean')
).reindex(order_clusters)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars1 = axes[0].barh(cluster_summary.index, cluster_summary['N_Usuarios'],
                      color=palette_c, edgecolor='white', linewidth=1.2, height=0.5)
axes[0].set_title('Cantidad de Usuarios por Perfil', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Número de usuarios')
axes[0].grid(axis='x', alpha=0.4)
for bar in bars1:
    width = bar.get_width()
    axes[0].annotate(f'{int(width):,}',
                     xy=(width, bar.get_y() + bar.get_height() / 2),
                     xytext=(6, 0), textcoords='offset points',
                     ha='left', va='center', fontsize=11, fontweight='bold')

bars2 = axes[1].barh(cluster_summary.index, cluster_summary['Gasto_Promedio'],
                      color=palette_c, edgecolor='white', linewidth=1.2, height=0.5)
axes[1].set_title('Gasto Promedio por Transacción ($)', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Gasto promedio ($)')
axes[1].grid(axis='x', alpha=0.4)
for bar in bars2:
    width = bar.get_width()
    axes[1].annotate(f'${width:,.0f}',
                     xy=(width, bar.get_y() + bar.get_height() / 2),
                     xytext=(6, 0), textcoords='offset points',
                     ha='left', va='center', fontsize=11, fontweight='bold')

plt.suptitle('Resumen de Clusters: Tamaño y Comportamiento de Gasto', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── GRÁFICA 2: Boxplot del comportamiento de compra por Cluster ───────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

vars_to_plot = ['Purchase_Mean', 'Purchase_Count', 'Purchase_Total']
titles = ['Gasto Promedio por Transacción ($)', 'Número de Transacciones', 'Gasto Total ($)']
palette = {'Comprador Ocasional': '#4C72B0', 'Comprador Frecuente': '#DD8452', 'Comprador VIP': '#55A868'}

for ax, var, title in zip(axes, vars_to_plot, titles):
    sns.boxplot(data=user_data, x='Cluster_Label', y=var, ax=ax, palette=palette,
                order=order_clusters)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=15)

plt.suptitle('Distribución del Comportamiento de Compra por Perfil', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── GRÁFICA 3: Perfil demográfico por Cluster ─────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
palette_list = ['#4C72B0', '#DD8452', '#55A868']

age_mid = {0: 13, 1: 21, 2: 30, 3: 40, 4: 48, 5: 53, 6: 60}
user_data['Age_Numeric'] = user_data['Age'].map(age_mid).fillna(user_data['Age'] * 5 + 13)

sns.barplot(data=user_data, x='Cluster_Label', y='Age_Numeric', ax=axes[0],
            palette=dict(zip(order_clusters, palette_list)), order=order_clusters, errorbar='sd')
axes[0].set_title('Edad Promedio por Perfil', fontsize=12, fontweight='bold')
axes[0].set_xlabel('')
axes[0].set_ylabel('Edad aproximada')
axes[0].tick_params(axis='x', rotation=15)

gender_cluster = user_data.groupby(['Cluster_Label', 'Gender']).size().unstack(fill_value=0)
gender_cluster = gender_cluster.div(gender_cluster.sum(axis=1), axis=0) * 100
gender_cluster = gender_cluster.reindex(order_clusters)
gender_cluster.columns = ['Femenino' if c == 0 else 'Masculino' for c in gender_cluster.columns]
gender_cluster.plot(kind='bar', stacked=True, ax=axes[1], color=['#C44E52', '#4C72B0'])
axes[1].set_title('Proporción de Género por Perfil', fontsize=12, fontweight='bold')
axes[1].set_xlabel('')
axes[1].set_ylabel('%')
axes[1].tick_params(axis='x', rotation=15)
axes[1].legend(title='Género')

city_cluster = user_data.groupby(['Cluster_Label', 'City_Category']).size().unstack(fill_value=0)
city_cluster = city_cluster.div(city_cluster.sum(axis=1), axis=0) * 100
city_cluster = city_cluster.reindex(order_clusters)
city_cluster.columns = [f'Ciudad {chr(65+c)}' for c in city_cluster.columns]
city_cluster.plot(kind='bar', stacked=True, ax=axes[2], colormap='tab10')
axes[2].set_title('Proporción de Ciudad por Perfil', fontsize=12, fontweight='bold')
axes[2].set_xlabel('')
axes[2].set_ylabel('%')
axes[2].tick_params(axis='x', rotation=15)
axes[2].legend(title='Ciudad')

plt.suptitle('Perfil Demográfico por Cluster', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── GRÁFICA 4: Scatter Plot PCA (reducción a 2D) ──────────────────────────────
from sklearn.decomposition import PCA

pca = PCA(n_components=2, random_state=SEED)
user_pca = pca.fit_transform(user_scaled)
explained = pca.explained_variance_ratio_

plt.figure(figsize=(10, 7))
colors_map = {
    'Comprador Ocasional': '#4C72B0',
    'Comprador Frecuente': '#DD8452',
    'Comprador VIP': '#55A868'
}
for label, color in colors_map.items():
    mask = user_data['Cluster_Label'] == label
    plt.scatter(user_pca[mask, 0], user_pca[mask, 1],
                label=label, color=color, alpha=0.5, s=20, edgecolors='none')

centroids_pca = pca.transform(kmeans.cluster_centers_)
for i, (cx, cy) in enumerate(centroids_pca):
    label_c = cluster_labels_map.get(i, f'Cluster {i}')
    color_c = colors_map.get(label_c, 'black')
    plt.scatter(cx, cy, marker='*', s=400, color=color_c,
                edgecolors='black', linewidths=0.8, zorder=5)

plt.xlabel(f'PC1 ({explained[0]*100:.1f}% varianza explicada)', fontsize=11)
plt.ylabel(f'PC2 ({explained[1]*100:.1f}% varianza explicada)', fontsize=11)
plt.title('Clusters de Usuarios en Espacio PCA (2D)', fontsize=14, fontweight='bold')
plt.legend(title='Perfil de Cluster', fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(f"Varianza total explicada por PC1+PC2: {(explained[0]+explained[1])*100:.1f}%")

In [ ]:
# ── GRÁFICA 5: Heatmap de perfil normalizado ──────────────────────────────────
keys_heatmap = ['Purchase_Mean', 'Purchase_Count', 'Purchase_Total',
                 'Pct_Expensive_Products', 'Unique_Products', 'Avg_Categories', 'Avg_Product_Price']
col_labels = ['Gasto Prom.', 'N Compras', 'Gasto Total',
              '% Prod. Caros', 'Prod. Únicos', 'Categ. Prom.', 'Precio Prom. Prod.']

cluster_profile = user_data.groupby('Cluster_Label')[keys_heatmap].mean()
cluster_profile.columns = col_labels
cluster_profile = cluster_profile.reindex(['Comprador Ocasional', 'Comprador Frecuente', 'Comprador VIP'])

cluster_norm = (cluster_profile - cluster_profile.min()) / (cluster_profile.max() - cluster_profile.min())

plt.figure(figsize=(13, 4))
sns.heatmap(cluster_norm,
            annot=cluster_profile.round(1),
            fmt='.1f',
            cmap='YlOrRd',
            linewidths=0.5,
            linecolor='white',
            cbar_kws={'label': 'Valor normalizado (0-1)'})
plt.title('Heatmap de Perfil de Clusters (valores reales anotados, intensidad normalizada)',
          fontsize=13, fontweight='bold')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# ── GRÁFICA 6: Radar Chart comparativo de perfiles ────────────────────────────
categorias = ['Gasto\nPromedio', 'N de\nCompras', 'Gasto\nTotal', '% Prod.\nCaros', 'Productos\nÚnicos']
keys_radar = ['Purchase_Mean', 'Purchase_Count', 'Purchase_Total', 'Pct_Expensive_Products', 'Unique_Products']

cluster_radar = user_data.groupby('Cluster_Label')[keys_radar].mean()
cluster_radar = cluster_radar.reindex(['Comprador Ocasional', 'Comprador Frecuente', 'Comprador VIP'])
cluster_radar_norm = (cluster_radar - cluster_radar.min()) / (cluster_radar.max() - cluster_radar.min())

N = len(categorias)
angles = [n / float(N) * 2 * np.pi for n in range(N)]
angles += angles[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw=dict(polar=True))
ax.set_theta_offset(np.pi / 2)
ax.set_theta_direction(-1)
ax.set_xticks(angles[:-1])
ax.set_xticklabels(categorias, fontsize=10)

colors_radar = ['#4C72B0', '#DD8452', '#55A868']
for (label, row), color in zip(cluster_radar_norm.iterrows(), colors_radar):
    values = row.tolist() + row.tolist()[:1]
    ax.plot(angles, values, linewidth=2, linestyle='solid', label=label, color=color)
    ax.fill(angles, values, alpha=0.15, color=color)

ax.set_ylim(0, 1)
ax.set_yticks([0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels(['25%', '50%', '75%', '100%'], fontsize=8)
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.15), title='Perfil')
plt.title('Radar Chart: Perfil Comparativo de Clusters', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.show()

---
### Interpretación y Conclusiones de los Clusters

A partir del análisis no supervisado con KMeans (K=3), identificamos tres perfiles de cliente bien diferenciados:

| Perfil | Gasto Promedio | Frecuencia | Gasto Total | Estrategia de Negocio |
|---|---|---|---|---|
| **Comprador Ocasional** | Bajo | Bajo | Bajo | Retención y activación: descuentos de bienvenida y cupones. |
| **Comprador Frecuente** | Medio | Alto | Alto | Fidelización: programas de puntos y cross-selling. |
| **Comprador VIP** | Alto | Medio | Alto | Experiencia premium: acceso anticipado, ofertas exclusivas. |

**Hallazgos clave:**
- El **porcentaje de productos caros** (`Pct_Expensive_Products`) es el factor más discriminante entre Ocasionales y VIP.
- El **número de transacciones** diferencia claramente a Compradores Frecuentes del resto.
- La **edad y el género** tienen menor poder discriminante, lo que sugiere que los patrones de compra trascienden la demografía.

### Conclusiones de Negocio Generales
1. **Regresión (Supervisado):** LightGBM con hiperparámetros optimizados es el modelo ganador. Las variables de ingeniería basadas en el historial del producto (`Product_Mean_Purchase`) y del usuario (`User_Mean_Purchase`) son los predictores más potentes. Al corregir el data leakage, las métricas reflejan el rendimiento real del modelo.
2. **Clustering (No Supervisado):** Los 3 perfiles identificados permiten diseñar estrategias de marketing diferenciadas. La combinación de frecuencia de compra y ticket promedio es suficiente para segmentar robustamente a los clientes de Black Friday.
